# 4.2.2 Clustering Stromal fraction

**Inés Sentís**

Date of execution

In [ ]:
Sys.Date()

## Libraries

In [ ]:
suppressPackageStartupMessages({
library(Seurat)
library(Signac)
library(tidyverse)
library(harmony)
library(scater) 
library(scran)
library(dittoSeq)
library(here)
library(glue)})

## Parameters

In [ ]:
#here::dr_here(show_reason = TRUE)
source(here("utils/bin.R"))
source(here("utils/paths.R"))
source(here("AVGUSTINOVA_02_to_30/utils/style.R"))
source(here("AVGUSTINOVA_02_to_30/utils/variables.R"))

"{int}/{robj_dir}" %>%
    glue::glue() %>%
    here::here() %>%
    dir.create(path = .,
               showWarnings = FALSE,
               recursive = TRUE)
set.seed(0)

## Load Data

In [ ]:
combined_obj <- readRDS(here(glue("{int}/{robj_dir}/4.2.2-stromal_fraction.rds")))

### High variable features

Remove genes that might influence biological variability

In [ ]:
genes_dmf <- c("XACT", "HTR2C",
               "XIST", "DDX3Y","USP9Y","NLGN4Y", "KDM5D", "EIF1AY", "UBA1")
mt_genes <- grep("^MT-", rownames(combined_obj[["RNA"]]), value = TRUE)
genes_out <- c(genes_dmf, top_c, mt_genes) 

In [ ]:
list_objects <- SplitObject(combined_obj, split.by = "patient")

In [ ]:
find_hvgs <-  function(obj) {
    
    pat <- unique(obj$patient)
    
    # remove genes from matrix before modulating variance and mean
    DefaultAssay(obj) <- "RNA"
    df_meta <- obj@meta.data
    counts <- obj[['RNA']]@counts
    counts <- counts[-(which(rownames(counts) %in% genes_out)),]
    obj2 <- CreateSeuratObject(counts = counts, meta.data = df_meta)
    
    # normalize again
    obj2 <- obj2 %>% 
            NormalizeData(
                normalization.method = "LogNormalize",
                scale.factor = 1e4) 
    
    # compute the variance and mean of the log-expression values 
    sce <- as.SingleCellExperiment(obj2)
    gene_var <- modelGeneVar(sce)
    
    tops <- gene_var %>% 
        as.data.frame() %>% 
        arrange(desc(total)) %>% 
        head(n=20)

    plt <- gene_var %>% 
      as.data.frame() %>% 
      ggplot(aes(mean, total)) +
      geom_point() +
      geom_line(aes(y = tech), colour = "dodgerblue", size = 1) +
      labs(x = "Mean of log-expression", y = "Variance of log-expression")+
      geom_text(data=tops, aes(mean,total,label=rownames(tops))) + 
      ggtitle(pat) + 
      theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold"))

    hvgs <- getTopHVGs(gene_var,fdr.threshold = 0.05)
    return(list(plot = plt, hvgs_pat = hvgs))
}

In [ ]:
modelling <- lapply(list_objects, find_hvgs)

In [ ]:
list_plots <- lapply(modelling, function(el) {
    el$plot
})

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 15)
cowplot::plot_grid(plotlist = list_plots,
                   align = "hv",
                   axis = "trbl",
                   ncol=4)

In [ ]:
list_hvg <- lapply(modelling, function(el) {
    hvg <- el$hvgs_pat
    print(length(hvg))
    hvg
})

In [ ]:
hvg_common <- find_perc(list_hvg, perc = 2/11)
length(hvg_common)

### Normalization and linear dimensional reduction

In [ ]:
VariableFeatures(combined_obj) <- hvg_common
 
combined_obj <- combined_obj %>% 
  NormalizeData(normalization.method = "LogNormalize",
                scale.factor = 1e4) %>%
  ScaleData(features=hvg_common) %>% 
  RunPCA(features=hvg_common) 

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 8)
ElbowPlot(combined_obj, ndims=50)

In [ ]:
rna_comp = 1:25

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)

DimPlot(combined_obj, reduction = "pca", group.by = c("clusters_donor","patient"), order=T, label=T)

In [ ]:
combined_obj <- RunUMAP(
  combined_obj,
  dims = rna_comp,
  reduction = "pca",
  reduction.name = "umap",
  reduction.key = "UMAP_"
)

## Integration with Harmony

In [ ]:
# make a copy of the PCA reduction
combined_obj@reductions$pca_filt <- combined_obj@reductions$pca

# Filter to the desired number of components according to elbow
combined_obj@reductions$pca_filt@cell.embeddings <- combined_obj@reductions$pca_filt@cell.embeddings[, rna_comp]

combined_obj@reductions$pca_filt@feature.loadings <- combined_obj@reductions$pca_filt@feature.loadings[, rna_comp]


combined_obj <- RunHarmony(
  object = combined_obj,
  reduction = "pca_filt",
  dims = rna_comp,
  group.by.vars = c("patient"),
  assay.use = "RNA",
  max_iter = 15,
  project.dim = T,
  reduction.save = "harmony_pca"
)

In [ ]:
combined_obj <- RunUMAP(
  combined_obj,
  dims = rna_comp,
  reduction = "harmony_pca",
  reduction.name = "umap.rna.harmony",
  reduction.key = "RNAhar_",
  assay = "RNA")

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 8)
DimPlot(
   combined_obj,
   reduction = "umap.rna.harmony",
   group.by = "patient",
   pt.size = 0.05,
   cols= colors_patients,
) + ggtitle('RNA UMAP\nby patient') + 
    theme(plot.title = element_text(hjust = 0.5, size = 12, face = "bold")) 

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10)
    FeaturePlot(combined_obj, reduction = "umap.rna.harmony",
                features = c("COL1A1","PECAM1","PODXL", "CAV1", "LAMA2", "PDGFRB"), 
                ncol=4, min.cutoff = "q05", max.cutoff = "q95")

## Clustering

In [ ]:
# Find the K-nearest neighbors
combined_obj <- FindNeighbors(combined_obj, reduction = "harmony_pca", 
                            dims = rna_comp)

In [ ]:
combined_obj <- FindClusters(combined_obj, resolution = 0.3, graph.name = "RNA_snn")

In [ ]:
options(repr.plot.width = 9, repr.plot.height = 8)
DimPlot(combined_obj, reduction = "umap.rna.harmony", label=T) +
    labs(title="Clustering with Harmony") + 
    theme(plot.title = element_text(hjust = 0.3, size = 12, face = "bold"))

In [ ]:
combined_obj <- FindSubCluster(
  object=combined_obj,
  cluster="4",
  graph.name="RNA_snn",
  subcluster.name = "subclustering",
  resolution = 0.2,
  algorithm = 1
)

DimPlot(combined_obj, reduction = "umap.rna.harmony", label=T,
                  group.by=glue::glue("subclustering")) +
    labs(title=glue::glue("Subclustering"))

In [ ]:
combined_obj <- SetIdent(combined_obj, value=combined_obj@meta.data[["subclustering"]])

In [ ]:
rna.markers <- FindAllMarkers(combined_obj, max.cells.per.ident=1000,
                              logfc.threshold = 0.1, only.pos = TRUE)

In [ ]:
rna.markers <- rna.markers %>% 
  dplyr::filter(p_val_adj < 0.05) %>% 
  arrange(desc(avg_log2FC))

In [ ]:
c="4_1"
rna.markers %>% 
    filter(cluster == c) %>% 
    head(n=50)

In [ ]:
rel <- dittoBarPlot(combined_obj, "clusters_donor", group.by = "ident", 
                      retain.factor.levels = TRUE) + 
            labs(title = "Relative proportion of \n clusters cells per patient") + 
            theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
            text =  element_text(hjust = 0.5,size = 14))

abs <- dittoBarPlot(combined_obj, "clusters_donor", group.by = "ident", 
            scale = "count", retain.factor.levels = TRUE) + 
            labs(title ="Absolute proportion of \n clusters cells per patient") + 
            theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
            text =  element_text(hjust = 0.5, size = 14))+ NoLegend()

In [ ]:
options(repr.plot.width = 12, repr.plot.height = 6, warn=-1,verbose = FALSE)
abs+rel

In [ ]:
# custom order for further vizualisations
combined_obj$patient <- factor(combined_obj$patient,
                                 levels=c("p19","p4","p15","p16","p10","p6","p3",
                                          "p5","p9","p13","p17","p14","p7","p8","p2"))

levels(combined_obj) <- c("0", "1", "2", "3","6","5","4_0","4_1","7")

In [ ]:
rename_clusters <- c("0"="c0:CAF SULF1",
                     "1"="c1:Myofibroblast",
                     "2"="c2:Mural cells", 
                     "3"="c3:Endothelial cells",
                     "6"="c6:Lymphatic endothelial",
                     "5"="c5:Adipocytes", 
                     "4_1"="c4_1:Neurons",
                     "4_0"="c4_0:Oligodendrocytes",  
                     "7"="c7:Astrocytes")
combined_obj <- RenameIdents(object = combined_obj, rename_clusters)
combined_obj$stromal_l2_clusters <- Idents(combined_obj)

In [ ]:
color_stromal = list(
  "c0:CAF SULF1" = "#0b6623",
  "c1:Myofibroblast" = "#228b22",
  "c2:Mural cells" = "#37c871ff",
  "c3:Endothelial cells" = "#9bdc65",
  "c6:Lymphatic endothelial" = "#b5f7a7",
  "c5:Adipocytes" = "#ffd54a",
  "c4_1:Neurons" = "#9ebcda",
  "c4_0:Oligodendrocytes" = "#7f2affff",
  "c7:Astrocytes" = "#c2a5cf"
)

In [ ]:
generate_color <- function(names_list) {
  colors <- ifelse(names_list %in% c( "c0:CAF SULF1",
                                     "c1:Myofibroblast",
                                    "c4_O:Oligodendrocytes"), "white", "black")
  return(colors)
}

color_labels <- generate_color(unique(Idents(combined_obj)))

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 6)
p_clust <- DimPlot(combined_obj, 
        reduction = "umap.rna.harmony",
        group.by="stromal_l2_clusters",
        label = T,
        label.box = TRUE,
        label.color = as.list(color_labels),
        label.size = 4.5,
        repel = TRUE,
        cols = unlist(color_stromal)) + 
    ggtitle("Stromal Clusters") + NoLegend() +
    theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"))

p_clust

In [ ]:
genes <- c("SULF1", "CDH11", "LAMA2", "VCAN", "TIMP1", "COL1A1","IGFBP2", "TAGLN", "ACTA2", "RGS5", "ABCC9",
          "FLT1", "VWF", "PLVAP","ACKR1","PECAM1","CCL21","PROX1","FLT4","GPAM","PPARG","ME1","LPL", 
          "NXPH1","NRXN1","INPP4B","SOX2-OT","CNTNAP2","NLGN1","ST18","MBP","PLP1","EYA1","PAX2","GFAP","AQP4","SLC1A2")

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 5)
dp1 <- dotplot_marker(combined_obj, genes, TRUE,"Stromal Clusters")
dp1

In [ ]:
rel <- dittoBarPlot(combined_obj, "stromal_l2_clusters", group.by = "patient", color.panel = color_stromal,
                      retain.factor.levels = FALSE,
                      var.labels.reorder = c(1,2,3,4,8,7,6,5,9)) + 
            labs(title = "Relative proportion of \n clusters cells per patient") + 
            theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
            text =  element_text(hjust = 0.5,size = 14, color = "black"))

abs <- dittoBarPlot(combined_obj, "stromal_l2_clusters", group.by = "patient", color.panel = color_stromal,
            scale = "count", retain.factor.levels = FALSE,
                    var.labels.reorder = c(1,2,3,4,8,7,6,5,9)) + 
            labs(title ="Absolute proportion of \n clusters cells per patient") + 
            theme(plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
            text =  element_text(hjust = 0.5, size = 14,color = "black"))+ NoLegend()

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 6, warn=-1,verbose = FALSE)
bp <- abs+rel
bp

## Save

In [ ]:
write_csv(rna.markers, here(glue("{int}/{robj_dir}/4.2.3-stromal_fraction_markers.csv")))

In [ ]:
source(here::here("utils/seurat2shiny.R"))
seurat2shiny(combined_obj, tech = "sc", reduction="umap.rna.harmony",
             save = T, path = here::here(glue::glue("{int}/{robj_dir}/")))

In [ ]:
ggsave(plot = p_clust, width = 7, height = 6, dpi = 300, 
       file = here::here(glue::glue("{int}/{plt_dir}/4.2.3-stromal_fraction_UMAP.svg")))

In [ ]:
ggsave(plot = dp1, width = 13, height = 5, dpi = 300, 
       file = here::here(glue::glue("{int}/{plt_dir}/4.2.3-stromal_fraction_dotplot.svg")))

In [ ]:
ggsave(plot = bp, width = 10, height = 6, dpi = 300, 
       file = here::here(glue::glue("{int}/{plt_dir}/4.2.3-stromal_fraction_barplot.pdf")))

In [ ]:
saveRDS(combined_obj, here(glue("{int}/{robj_dir}/4.2.3-stromal_fraction.rds")))

In [ ]:
sessionInfo()